# Paper-figure parity: logit lens, cluster map, component loadings

Three figure replications from the Anthropic emotions paper that did not yet have a home:
Table 1 (logit lens), Figure 6 (emotion cluster map), Figure 7 (principal-component loading
bars). The full inventory with per-figure status lives in `notes/plot_parity.md`.

Data: base-model emotion means (`results/emotion_vectors/emotion_means.npz`, the arm where the
geometry replicates) and the corrected logit-lens output (`results/logit_lens_it_L57.json`).

## 1. Logit lens (paper Table 1): does not reproduce on the instruct model

The paper projects each emotion vector through the unembedding matrix and finds emotion-word
neighborhoods (sad up-weights grief, tears, lonely). We ran the same projection on the
instruct-model vectors, with the final normalization layer's scaling applied.

How to read: each row lists the five most up-weighted tokens. The paper's result would show
emotion-related words. Ours shows unrelated fragments, a robust negative across layers 33 and
57, with and without norm scaling. Documented caveats: Gemma's logit softcapping is ignored,
and the base-model arm (where geometry replicates) still needs its own run; it requires
reloading the evicted base model and is queued.

In [1]:
# this cell renders the corrected logit-lens table for the instruct model
import json
from pathlib import Path

import numpy as np
import plotly.graph_objects as go

ROOT = Path("..")
lens = json.load(open(ROOT / "results/logit_lens_it_L57.json"))
rows = [(e, ", ".join(t["up"]), ", ".join(t["down"])) for e, t in lens["table"].items()]
fig = go.Figure(go.Table(
    header=dict(values=["emotion", "top up-weighted tokens", "top down-weighted tokens"],
                align="left"),
    cells=dict(values=list(zip(*rows)), align="left", height=26),
))
fig.update_layout(title=f"Logit lens, gemma-4-31b-it, layer {lens['layer']} ({lens['note']})",
                  height=460, margin=dict(t=50, b=10))
fig.show()
print("verdict: no emotion-word neighborhoods; paper Table 1 does not reproduce on -it vectors")

verdict: no emotion-word neighborhoods; paper Table 1 does not reproduce on -it vectors


## 2. Emotion cluster map (paper Figure 6, embedding substituted)

The paper clusters emotion vectors with k-means and displays them with UMAP (uniform manifold
approximation and projection). We substitute t-SNE (t-distributed stochastic neighbor
embedding, available in scikit-learn) to avoid an extra dependency; the clustering itself is
identical (k-means, k=10). Base model, layer 33, centered vectors.

How to read: each point is one of the 171 emotions, colored by its k-means cluster. The
paper's result, interpretable clusters (joy-family together, grief-family together), should
appear as color-coherent neighborhoods with sensible members.

In [2]:
# this cell clusters base-model emotion vectors and embeds them for display
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE

bundle = np.load(ROOT / "results/emotion_vectors/emotion_means.npz", allow_pickle=True)
emotions = list(map(str, bundle["emotions"]))
layers = list(bundle["layers"])
M = bundle["means"][:, layers.index(33), :].astype(np.float64)
M -= M.mean(axis=0)

km = KMeans(n_clusters=10, n_init=10, random_state=20260722).fit(M)
xy = TSNE(n_components=2, perplexity=20, random_state=20260722).fit_transform(M)

fig = go.Figure()
for c in range(10):
    idx = np.where(km.labels_ == c)[0]
    fig.add_scatter(x=xy[idx, 0], y=xy[idx, 1], mode="markers+text",
                    text=[emotions[i] for i in idx], textposition="top center",
                    textfont=dict(size=7), name=f"cluster {c}",
                    marker=dict(size=7))
fig.update_layout(title="Emotion clusters, base model layer 33 (k-means k=10, t-SNE embedding)",
                  xaxis_title="t-SNE dim 1", yaxis_title="t-SNE dim 2", height=650,
                  showlegend=False)
fig.show()

for c in range(10):
    members = [emotions[i] for i in np.where(km.labels_ == c)[0]][:8]
    print(f"cluster {c}: {members}")

cluster 0: ['blissful', 'enthusiastic', 'grateful', 'happy', 'hopeful', 'inspired', 'invigorated', 'optimistic']
cluster 1: ['anxious', 'awestruck', 'bewildered', 'desperate', 'disoriented', 'dispirited', 'distressed', 'disturbed']
cluster 2: ['at ease', 'calm', 'content', 'fulfilled', 'hope', 'loving', 'patient', 'peaceful']
cluster 3: ['disdainful', 'greedy', 'relieved', 'self-confident', 'stubborn', 'valiant']
cluster 4: ['afraid', 'bored', 'brooding', 'dependent', 'depressed', 'docile', 'droopy', 'empathetic']
cluster 5: ['alarmed', 'amazed', 'annoyed', 'aroused', 'ashamed', 'astonished', 'bitter', 'defiant']
cluster 6: ['alert', 'angry', 'contemptuous', 'dumbstruck', 'embarrassed', 'enraged', 'exasperated', 'furious']
cluster 7: ['ecstatic', 'elated', 'euphoric', 'playful']
cluster 8: ['amused', 'cheerful', 'delighted', 'energized', 'excited', 'exuberant', 'joyful', 'jubilant']
cluster 9: ['compassionate', 'envious', 'indifferent', 'infatuated', 'jealous', 'lazy', 'lonely', 'paran

## 3. Component loadings (paper Figure 7)

How to read: bars are each emotion's score on the first principal component (top panel,
expected to order negative to positive feelings) and the second (bottom panel, expected to
separate low-arousal from high-arousal states). Sparse labels, as in the paper. Signs are
orientation-arbitrary; both components are oriented so their correlate (valence, arousal)
increases to the right.

In [3]:
# this cell computes PCA loadings and draws the two ordered bar panels
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA

from emotion_vectors.analysis import load_nrc_vad

vad = load_nrc_vad(ROOT / "data/lexicons/NRC-VAD-Lexicon-v2.1/NRC-VAD-Lexicon-v2.1.txt")
matched = [i for i, e in enumerate(emotions) if e.lower() in vad]
valence = np.array([vad[emotions[i].lower()][0] for i in matched])
arousal = np.array([vad[emotions[i].lower()][1] for i in matched])

pca = PCA(n_components=2)
scores = pca.fit_transform(M)
if np.corrcoef(scores[matched, 0], valence)[0, 1] < 0:
    scores[:, 0] *= -1
if np.corrcoef(scores[matched, 1], arousal)[0, 1] < 0:
    scores[:, 1] *= -1

fig = make_subplots(rows=2, cols=1, subplot_titles=(
    f"first principal component ({pca.explained_variance_ratio_[0]:.0%} variance, tracks valence)",
    f"second principal component ({pca.explained_variance_ratio_[1]:.0%} variance, tracks arousal)"))
for row, comp in ((1, 0), (2, 1)):
    order = np.argsort(scores[:, comp])
    labels = [emotions[i] if r % 8 == 0 else "" for r, i in enumerate(order)]
    fig.add_bar(x=list(range(len(order))), y=scores[order, comp], row=row, col=1,
                marker_color=scores[order, comp], marker_colorscale="RdYlGn",
                showlegend=False)
    fig.update_xaxes(tickvals=list(range(len(order))), ticktext=labels, tickangle=60,
                     tickfont=dict(size=7), row=row, col=1)
fig.update_layout(title="Emotion loadings on the top two principal components (base, layer 33)",
                  height=700)
fig.show()

## Status

Updates `notes/plot_parity.md`: Figure 6 substituted and done, Figure 7 done, Table 1
attempted with a documented negative on the instruct arm and a queued base-arm run. Remaining
queued items: Figure 1 (scaled corpus sweep) and the token-level appendix visualization,
which shares infrastructure with the planned temporal-dynamics work.